# Multivariate fine-mapping (mvSuSiE)

## Description

Per-region multivariate fine-mapping via `pecotmr::fineMappingPipeline(qtlDataset, methods = "mvsusie")`. mvSuSiE jointly fine-maps across the QtlDataset's traits and contexts in one region, producing one `QtlFineMappingResult` per fan-out unit. Each task loads the same `QtlDataset` RDS and fits one unit.

This replaces the `[mnm]` and `[mnm_genes]` steps of the legacy `mnm_regression.ipynb`. The same `fine_mapping.R` worker that drives univariate SuSiE drives mvSuSiE here — only the `--methods` value differs.

For mvSuSiE TWAS weights (mr.mash + mvSuSiE), call `twas_weights.ipynb --methods mrmash,mvsusie` afterwards against the same `QtlDataset`.

## Inputs

- `--qtl-dataset` &mdash; path to the `QtlDataset` RDS produced by `qtl_dataset.ipynb`. Multi-context / multi-trait shape is what makes mvSuSiE meaningful.
- `--regions chr:start-end ...` **or** `--genes ID1 ID2 ...` &mdash; fan-out targets (mutually exclusive). Region mode (multi-trait joint) is the standard use; gene mode runs the multi-context mvSuSiE variant for one focal trait.
- `--cis-window` &mdash; bp window (gene mode only). Default 1,000,000.
- `--methods` &mdash; comma-separated method tokens. Default `mvsusie`. Pass e.g. `mvsusie,susie` to also run univariate SuSiE alongside.
- `--coverage` &mdash; SuSiE/mvSuSiE credible-set coverage. Default 0.95.
- `--method-args` &mdash; optional JSON object spliced into `fineMappingPipeline()` for knobs not exposed as flags (priors, max-iter, residual options, etc.).
- `--cwd` &mdash; output directory. Default `output`.
- `--study` &mdash; study label used in the output filename.
- `--modular-script-dir` &mdash; directory holding the per-task R workers. Default `code/script`.

## Outputs

- `{cwd}/{study}.{gene|region}.mv_finemap.rds` &mdash; one `QtlFineMappingResult` per fan-out unit. Region strings are sanitised (`:` and `-` become `_`).


## Example

Region mode (canonical, multi-trait joint):
```bash
sos run pipeline/multivariate_fine_mapping.ipynb multivariate_fine_mapping \
    --cwd output \
    --modular-script-dir /path/to/xqtl-protocol/code/script \
    --study TEST_STUDY \
    --qtl-dataset output/TEST_STUDY.qtl_dataset.rds \
    --regions chr12:752578-2752578
```

With mixture-prior tuning passed through:
```bash
sos run pipeline/multivariate_fine_mapping.ipynb multivariate_fine_mapping \
    --cwd output --study TEST_STUDY \
    --qtl-dataset output/TEST_STUDY.qtl_dataset.rds \
    --regions chr12:752578-2752578 \
    --method-args '{"addSusieInf":false,"signalCutoff":0.01}'
```


In [ ]:
[global]
parameter: cwd = path('output')
parameter: study = str
parameter: modular_script_dir = path('code/script')
parameter: qtl_dataset = path('.')
parameter: genes = []
parameter: regions = []
parameter: cis_window = 1000000
parameter: methods = 'mvsusie'
parameter: coverage = 0.95
# JSON object spliced into fineMappingPipeline() via do.call. Empty disables.
parameter: method_args = ''
parameter: container = ''
parameter: job_size = 1
parameter: walltime = '1h'
parameter: mem = '16G'
parameter: numThreads = 1

In [ ]:
[multivariate_fine_mapping]
if not qtl_dataset.is_file():
    raise ValueError("multivariate_fine_mapping requires --qtl-dataset to point at an existing QtlDataset RDS.")
if bool(genes) == bool(regions):
    raise ValueError("Specify exactly one of --genes (trait IDs) or --regions (chr:start-end strings).")
fanout_items = genes if genes else regions
fanout_kind  = 'gene' if genes else 'region'
input: qtl_dataset, for_each = 'fanout_items'
output: f"{cwd}/{study}.{_fanout_items.replace(':', '_').replace('-', '_')}.mv_finemap.rds"
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f"{step_name}_{_output:bn}"
bash: expand = '${ }', stderr = f"{_output}.stderr", stdout = f"{_output}.stdout", container = container
    Rscript ${modular_script_dir}/pecotmr_integration/fine_mapping.R \
        --qtl-dataset ${_input} \
        ${('--gene-id ' + _fanout_items + ' --cis-window ' + str(cis_window)) if fanout_kind == 'gene' else ('--region ' + _fanout_items)} \
        --methods ${methods} \
        --coverage ${coverage} \
        ${('--method-args ' + repr(method_args)) if method_args else ''} \
        --output ${_output}